# 04 · Placas: detección (YOLOv8n) y lectura (OCR)

Se ejecuta en **Google Colab** con GPU (Entorno de ejecución → Cambiar tipo → T4), de arriba abajo.
Los pesos y las imágenes **no** se suben a git (I6); las métricas (`metrics.json`) sí se versionan en `ml/experiments/`.

Antes: genera el ZIP de la carpeta `ml/` (sin `data/`) y ten a mano la carpeta del repositorio.

In [ ]:
import os, zipfile
from google.colab import files

subidos = files.upload()          # selecciona solo ml.zip (carpeta ml/ del repo, sin data/)
nombre = list(subidos)[0]         # Colab renombra a "ml (1).zip" si ya existía: usa el nombre real
os.makedirs("/content/condohub-ia", exist_ok=True)
zipfile.ZipFile(nombre).extractall("/content/condohub-ia")
os.environ["ML_DATA_DIR"] = "/content/condohub-ml"
os.chdir("/content/condohub-ia")
print(os.listdir("/content/condohub-ia"))

In [ ]:
!pip install -q ultralytics easyocr kagglehub pyyaml
import torch
print("GPU disponible:", torch.cuda.is_available())

## 1 · Datos

Dataset público *Car License Plate Detection* (433 imágenes, una clase, CC0). Si la descarga automática falla
(Kaggle a veces pide iniciar sesión), descarga el ZIP a mano desde Kaggle y súbelo cuando se pida.

In [ ]:
import yaml, glob, shutil
from pathlib import Path

cfg = yaml.safe_load(open("ml/configs/placas_deteccion.yaml", encoding="utf-8"))
crudo = Path("/content/placas_crudo")

try:
    import kagglehub
    origen = Path(kagglehub.dataset_download(cfg["dataset"]["kaggle"]))
except Exception as e:
    print("Descarga automática no disponible:", type(e).__name__, "-> sube el ZIP del dataset")
    from google.colab import files
    z = list(files.upload())[0]
    shutil.rmtree(crudo, ignore_errors=True)
    zipfile.ZipFile(z).extractall(crudo)
    origen = crudo

imagenes = {p.stem: p for ext in ("png", "jpg", "jpeg") for p in origen.rglob(f"*.{ext}")}
anotaciones = {p.stem: p for p in origen.rglob("*.xml")}
comunes = sorted(set(imagenes) & set(anotaciones))
print(len(imagenes), "imágenes |", len(anotaciones), "anotaciones |", len(comunes), "con ambas")

In [ ]:
from ml.src import placas

partes = placas.dividir_imagenes(comunes, tuple(cfg["dataset"]["proporciones"]), cfg["semilla"])
raiz = Path("/content/placas")
shutil.rmtree(raiz, ignore_errors=True)

for parte, nombres in partes.items():
    (raiz / "images" / parte).mkdir(parents=True)
    (raiz / "labels" / parte).mkdir(parents=True)
    for n in nombres:
        lineas = placas.voc_a_yolo(anotaciones[n].read_text(encoding="utf-8"))
        if not lineas:
            continue
        shutil.copy(imagenes[n], raiz / "images" / parte / imagenes[n].name)
        (raiz / "labels" / parte / f"{n}.txt").write_text("\n".join(lineas), encoding="utf-8")

data_yaml = raiz / "data.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path": str(raiz), "train": "images/train", "val": "images/val", "test": "images/test",
    "names": {0: "placa"},
}), encoding="utf-8")
print({k: len(list((raiz / "images" / k).iterdir())) for k in partes})

## 2 · Entrenamiento (ajuste fino desde COCO)

YOLOv8n preentrenado en COCO **no tiene la clase «placa»**, así que no existe un detector preentrenado con el que
comparar: el «antes» es el modelo sin ajustar (mAP ≈ 0 en esta clase) y el «después» es el ajustado.

In [ ]:
from ultralytics import YOLO

modelo = YOLO(cfg["entrenamiento"]["modelo_inicial"])
_ = modelo.train(      # el "_" evita que Colab imprima el objeto de resultados completo
    data=str(data_yaml), epochs=cfg["entrenamiento"]["epocas"], imgsz=cfg["entrenamiento"]["tamano_imagen"],
    batch=cfg["entrenamiento"]["lote"], seed=cfg["semilla"], project="/content/runs", name="placas", exist_ok=True,
    patience=20, plots=True,
)

## 3 · Evaluación en la partición de **prueba** (imágenes que no se usaron para entrenar ni elegir el modelo)

In [ ]:
mejor = YOLO("/content/runs/placas/weights/best.pt")
r = mejor.val(data=str(data_yaml), split="test", imgsz=cfg["entrenamiento"]["tamano_imagen"])
deteccion = {
    "map50": float(r.box.map50), "map50_95": float(r.box.map),
    "precision": float(r.box.mp), "recall": float(r.box.mr),
    "imagenes_prueba": len(partes["test"]),
}
print(deteccion)

In [ ]:
import matplotlib.pyplot as plt

muestra = sorted((raiz / "images" / "test").iterdir())[:6]
fig, ejes = plt.subplots(2, 3, figsize=(14, 8))
for eje, ruta in zip(ejes.ravel(), muestra):
    eje.imshow(mejor.predict(str(ruta), verbose=False)[0].plot()[..., ::-1])
    eje.axis("off")
plt.show()

## 4 · Lectura de la placa (OCR)

Este dataset no trae el **texto** de las placas, así que la precisión por carácter se mide con placas
**sintéticas** de texto conocido (formato A4: 4 dígitos + 3 letras). Son optimistas frente a placas reales:
decláralo así en el informe. Se compara el OCR crudo contra dos pasos de posproceso del formato: extraer la ventana
`4 dígitos + 3 letras` (descarta el marco y otros sobrantes) y corregir letra/dígito por posición.

In [ ]:
import random, numpy as np, easyocr
from ml.src import placas_sinteticas

lector = easyocr.Reader(cfg["ocr"]["idioma"], gpu=torch.cuda.is_available())
rng = random.Random(cfg["semilla"])
reales, crudas, extraidas, corregidas = [], [], [], []
for _ in range(cfg["ocr"]["placas_sinteticas"]):
    texto = placas_sinteticas.texto_aleatorio(rng)
    imagen = placas_sinteticas.dibujar(texto, rng)
    bloques = lector.readtext(np.asarray(imagen), allowlist=cfg["ocr"]["permitidos"], detail=1)
    bloques.sort(key=lambda b: b[0][0][0])              # de izquierda a derecha, para no mezclar el orden
    leido = placas.normalizar("".join(b[1] for b in bloques).replace("BOLIVIA", ""))
    reales.append(texto)
    crudas.append(leido)
    extraidas.append(placas.extraer_placa(leido))
    corregidas.append(placas.corregir_por_posicion(placas.extraer_placa(leido)))

ocr_sintetico = {
    "crudo": placas.metricas_lectura(reales, crudas),
    "con_extraccion": placas.metricas_lectura(reales, extraidas),
    "con_extraccion_y_posicion": placas.metricas_lectura(reales, corregidas),
}
for nombre, m in ocr_sintetico.items():
    print(f"{nombre:<26}", m)
print("con formato válido (crudo):", sum(placas.es_placa_valida(c) for c in crudas), "de", len(crudas))
print("con formato válido (final):", sum(placas.es_placa_valida(c) for c in corregidas), "de", len(corregidas))
for r, c in zip(reales, corregidas):
    if r != c:
        print("falla:", r, "->", c)

## 5 · Guardar y descargar

Las métricas se copian a `ml/experiments/<fecha>_placas_deteccion/` del repo. Los pesos (`best.pt`) **no** van a git.

In [ ]:
import json
from datetime import date
from google.colab import files
from ml.src.paths import dir_datos, dir_experimento

destino = dir_experimento(cfg["nombre"])
resultado = {"config": cfg, "deteccion_prueba": deteccion,
             "ocr_sintetico": ocr_sintetico,
             "particion": {k: len(v) for k, v in partes.items()}}
(destino / "metrics.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8")

pesos = dir_datos() / "modelos" / "placas" / f"{date.today().isoformat()}_{cfg['nombre']}"
pesos.mkdir(parents=True, exist_ok=True)
shutil.copy("/content/runs/placas/weights/best.pt", pesos / "yolov8n_placas.pt")
shutil.copy("/content/runs/placas/results.png", destino / "curvas_entrenamiento.png")

files.download(str(destino / "metrics.json"))
files.download(str(destino / "curvas_entrenamiento.png"))
files.download(str(pesos / "yolov8n_placas.pt"))   # guárdalo fuera de git (p. ej. D:\SW2\modelos\placas)